# 05A_CNN_DataLoader.ipynb

Creates a PyTorch Dataset and DataLoaders from downloaded GeoTIFF images.

In [ ]:

!pip -q install rasterio torch torchvision scikit-learn

from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import rasterio
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

ROOT='/content/drive/MyDrive/FloodProject'
SENT=os.path.join(ROOT,'Sentinel')
LAND=os.path.join(ROOT,'Landsat')

IMAGE_SIZE=512
BATCH_SIZE=8

files=glob.glob(os.path.join(SENT,'*.tif'))+glob.glob(os.path.join(LAND,'*.tif'))
print('Images Found:',len(files))

class SatelliteDataset(Dataset):
    def __init__(self, files):
        self.files=files
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        fp=self.files[idx]
        with rasterio.open(fp) as src:
            img=src.read().astype(np.float32)
        img=np.clip(img/10000.0,0,1)
        x=torch.from_numpy(img)
        x=F.interpolate(x.unsqueeze(0),size=(IMAGE_SIZE,IMAGE_SIZE),
                        mode='bilinear',align_corners=False).squeeze(0)
        return {'image':x,'event_id':os.path.splitext(os.path.basename(fp))[0]}

dataset=SatelliteDataset(files)
train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size
train_ds,val_ds=random_split(dataset,[train_size,val_size],generator=torch.Generator().manual_seed(42))
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE)

batch=next(iter(train_loader))
print('Train:',len(train_ds))
print('Validation:',len(val_ds))
print('Image Tensor Shape:',batch['image'].shape)
print('Event IDs:',batch['event_id'])
